# 3.5 Email Agent — proyecto del módulo 3 (adaptado a lab local)

**Qué se ve aquí:** combina todo el módulo en un agente:
- **State** `authenticated` + **context** con email y contraseña.
- **Dynamic tools** (`@wrap_model_call`): sin autenticar solo ve `authenticate`; autenticado ve `check_inbox` y `send_email`.
- **Dynamic prompt** según el estado de autenticación.
- **HITL** antes de `send_email`.

También se sirve con `langgraph dev` (`3.5_email_agent.py` + `langgraph.json`) y se prueba en **Agent Chat UI** (`agent-chat-ui/`).

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

email agent
- authenticates user
    - only then are they allowed into the "inbox"
    - dynamic tools and prompt on the condition of there being an email and password in state that match hardcoded
- checks "inbox"
    - email in tool
- sends emails
    - human in the loop

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# import warnings
# warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-3/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)

import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest")

In [ ]:
# Sin cambios
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

In [ ]:
# Sin cambios
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

In [ ]:
# Sin cambios
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

In [ ]:
# Sin cambios
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

>Note: the prompts were modified since filming to constrain the model to more reliably match the filmed sequence. You may still experience different responses from the model, which is expected. You may need to modify the human message to provide appropriate responses.

In [ ]:
# Sin cambios
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = """You are a helpful assistant that can check the inbox and send emails. 
Your first step after authentication is to check the inbox."""
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
# from langgraph.checkpoint.memory import InMemorySaver
# from langchain.agents.middleware import HumanInTheLoopMiddleware
#
# agent = create_agent(
#     "gpt-5-nano",
#     tools=[authenticate, check_inbox, send_email],
#     checkpointer=InMemorySaver(),
#     state_schema=AuthenticatedState,
#     context_schema=EmailContext,
#     middleware=[
#         dynamic_tool_call, 
#         dynamic_prompt,
#         HumanInTheLoopMiddleware(
#             interrupt_on={
#                 "authenticate": False,
#                 "check_inbox": False,
#                 "send_email": True,
#             })
#         ]
#     )
#

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" es API de pago.
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    MODEL,
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )


In [ ]:
# Sin cambios
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

In [ ]:
# Sin cambios

response = agent.invoke(
    {"messages": [HumanMessage(content="any draft is fine. don't check back.")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

In [ ]:
# Sin cambios
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

In [ ]:
# Sin cambios
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

In [ ]:
# Sin cambios
from pprint import pprint

pprint(response)